In [0]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("Dynamic Partition Pruning Example").getOrCreate()



In [0]:
# Sample data for sales (fact table) partitioned by date
sales_data = [
    ("2024-01-01", "product1", 100),
    ("2024-01-02", "product2", 200),
    ("2024-01-01", "product3", 150),
    ("2024-01-03", "product1", 120),
    ("2024-01-02", "product4", 180),
]

# Sample data for products (dimension table)
product_data = [
    ("product1", "Electronics"),
    ("product2", "Clothing"),
    ("product3", "Books"),
    ("product4", "Sports"),
]


In [0]:
# Create DataFrames
sales_df = spark.createDataFrame(sales_data, ["date", "product", "sales"])
products_df = spark.createDataFrame(product_data, ["product", "category"])

In [0]:
# Write the fact table as a partitioned table
sales_df.write.partitionBy("date").mode("overwrite").parquet("/tmp/sales_data")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-6840436441391980>, line 2
      1 # Write the fact table as a partitioned table
----> 2 sales_df.write.partitionBy("date").mode("overwrite").parquet("/tmp/sales_data")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    70

In [0]:
# Read the partitioned sales data
sales_df_partitioned = spark.read.parquet("/tmp/sales_data")

In [0]:
from pyspark.sql import functions as F

# Filter products by category
filtered_products = products_df.filter(products_df.category == "Electronics")

# Perform the join between the partitioned sales table and filtered products
# Dynamic partition pruning will ensure only relevant partitions (dates) from sales_df are scanned.
joined_df = sales_df_partitioned.join(filtered_products, "product")

# Trigger the execution (e.g., show, collect)
joined_df.show()